# 第三章：查询引擎与响应合成

## 学习目标
- 理解查询引擎的内部流程：Retriever → PostProcessor → ResponseSynthesizer
- 控制检索参数（similarity_top_k 等）
- 掌握不同响应合成模式（compact / refine / tree_summarize）
- 自定义查询引擎的提示词模板
- 使用 as_retriever() 进行纯检索（不经过 LLM）

## 0. 环境初始化

复用前两章的配置：加载环境变量、初始化 LLM 和 Embedding 模型、设置全局 Settings。

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 初始化 LLM
llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from llama_index.llms.openai_like import OpenAILike
# llm = OpenAILike(model="glm-4-plus", api_base=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"], is_chat_model=True)

# 初始化 Embedding 模型
embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

# 全局配置
Settings.llm = llm
Settings.embed_model = embed_model

print(f"LLM: {Settings.llm.model}")
print(f"Embedding: {Settings.embed_model.model_name}")

In [ ]:
from llama_index.core import Document, VectorStoreIndex

# 构建示例文档（AI 技术相关知识）
documents = [
    Document(text="RAG（检索增强生成）是一种结合信息检索和文本生成的技术。它先从知识库中检索相关文档，再将检索结果作为上下文提供给 LLM，让模型基于真实数据生成回答，减少幻觉。", metadata={"topic": "RAG"}),
    Document(text="向量数据库是专门用于存储和检索高维向量的数据库。它通过将文本转换为向量（embedding），然后用相似度搜索找到语义最接近的内容。常见的向量数据库有 FAISS、Pinecone、Weaviate 等。", metadata={"topic": "向量数据库"}),
    Document(text="文本嵌入（Text Embedding）是将文本转换为固定维度的数值向量的过程。语义相近的文本在向量空间中距离更近。嵌入模型如 text-embedding-v3 可以将任意长度的文本映射为向量。", metadata={"topic": "嵌入"}),
    Document(text="提示词工程（Prompt Engineering）是设计和优化 LLM 输入提示的技术。好的提示词可以显著提升模型输出质量。常见技巧包括：角色设定、Few-shot 示例、思维链（CoT）等。", metadata={"topic": "提示词工程"}),
    Document(text="LLM 幻觉（Hallucination）是指模型生成看似合理但实际错误的内容。RAG 是减少幻觉的有效手段之一，因为它让模型基于检索到的真实文档回答问题，而不是完全依赖训练数据。", metadata={"topic": "幻觉"}),
    Document(text="Agent 是一种能够自主决策和执行任务的 AI 系统。它通过工具调用（Tool Calling）与外部系统交互，使用 ReAct 等模式循环执行'思考→行动→观察'直到任务完成。", metadata={"topic": "Agent"}),
]

index = VectorStoreIndex.from_documents(documents)
print(f"索引构建完成，包含 {len(documents)} 个文档")

### 刚才发生了什么？

我们用 6 个 AI 技术相关的文档构建了一个内存中的向量索引。每个文档都有 `metadata` 标注主题，方便后续观察检索效果。这个索引将贯穿本章所有示例。

## 1. 查询引擎的内部流程

当你调用 `index.as_query_engine()` 并执行 `.query()` 时，背后其实经历了三个步骤：

```
用户问题
   │
   ▼
┌─────────────────────────┐
│  ① Retriever（检索器）     │  将问题转为向量，找到 top-k 个最相似的节点
└─────────────────────────┘
   │
   ▼
┌─────────────────────────┐
│  ② Node Postprocessor    │  对检索结果做后处理：过滤、重排序、截断
│  （后处理器，可选）         │  （本章暂不涉及，第五章详讲）
└─────────────────────────┘
   │
   ▼
┌─────────────────────────┐
│  ③ Response Synthesizer  │  将检索到的内容 + 问题发送给 LLM
│  （响应合成器）             │  按照指定模式生成最终回答
└─────────────────────────┘
   │
   ▼
最终回答（Response 对象）
```

与 LangChain 的 RAG 链对比：

| 步骤 | LangChain | LlamaIndex |
|------|-----------|------------|
| 检索 | `vectorstore.as_retriever()` | `index.as_retriever()` |
| 后处理 | 需手动在 LCEL 中添加 | 内置 `node_postprocessors` 参数 |
| 合成 | 需手动构建 prompt + LLM 链 | 内置 `ResponseSynthesizer`，一个参数切换模式 |

> **关键区别**：LangChain 的 RAG 需要你自己编排检索→格式化→LLM 的链条（LCEL），LlamaIndex 把这三步封装成了 `QueryEngine`，开箱即用。

## 1.1 默认查询引擎

In [ ]:
# 默认查询引擎（使用所有默认参数）
query_engine = index.as_query_engine()
response = query_engine.query("什么是 RAG？它为什么能减少幻觉？")
print(response)

### 刚才发生了什么？

1. `as_query_engine()` 使用默认参数创建了一个查询引擎（`similarity_top_k=2`，`response_mode="compact"`）
2. `.query()` 内部先把问题向量化，检索出 2 个最相似的文档节点
3. 然后把这些节点内容和问题一起发送给 LLM，生成回答

但我们看不到中间过程——检索了哪些内容？相似度是多少？接下来拆开看。

In [ ]:
# 查看回答引用了哪些源文档
for i, node in enumerate(response.source_nodes):
    print(f"源节点 {i+1} (相似度: {node.score:.4f}):")
    print(f"  主题: {node.metadata.get('topic', '未知')}")
    print(f"  内容: {node.text[:60]}...")
    print()

### 刚才发生了什么？

`response.source_nodes` 包含了查询引擎检索到的所有源节点。每个节点都有：
- `score`：与查询的相似度分数
- `text`：节点文本内容
- `metadata`：元数据

这是 LlamaIndex 的一个优点——回答自带溯源信息，方便调试和验证。

## 2. 控制检索参数

`similarity_top_k` 控制检索多少个最相似的节点。数量越多，LLM 获得的上下文越丰富，但也可能引入噪声。

In [ ]:
# 对比不同的 top_k 设置
for k in [1, 3, 5]:
    engine = index.as_query_engine(similarity_top_k=k)
    response = engine.query("RAG 是什么？")
    print(f"--- top_k={k} ---")
    print(f"回答: {str(response)[:100]}...")
    print(f"使用了 {len(response.source_nodes)} 个源节点")
    print()

### 刚才发生了什么？

- `top_k=1`：只检索最相关的 1 个文档，回答信息有限
- `top_k=3`：检索 3 个文档，上下文更充分
- `top_k=5`：检索 5 个文档，可能包含不太相关的内容

**经验法则**：对于简单事实问题，`top_k=2~3` 通常足够；对于需要综合多个来源的复杂问题，可以适当增大。

### `as_query_engine()` 常用参数

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `similarity_top_k` | 检索最相似的 k 个节点 | 2 |
| `response_mode` | 响应合成模式 | `"compact"` |
| `streaming` | 是否流式输出 | `False` |
| `text_qa_template` | 自定义 QA 提示词模板 | 内置默认模板 |
| `node_postprocessors` | 后处理器列表 | `None` |

## 3. 纯检索模式（不经过 LLM）

有时候你只想看「检索到了什么」，不需要 LLM 生成回答。这在调试 RAG 效果时特别有用。

In [ ]:
# 只做检索，不调用 LLM
retriever = index.as_retriever(similarity_top_k=3)
nodes = retriever.retrieve("向量数据库有哪些？")

for i, node in enumerate(nodes):
    print(f"节点 {i+1} (相似度: {node.score:.4f}):")
    print(f"  主题: {node.metadata.get('topic', '未知')}")
    print(f"  内容: {node.text[:80]}...")
    print()

### 刚才发生了什么？

`as_retriever()` 返回一个纯检索器，调用 `.retrieve()` 只执行向量搜索，**不调用 LLM**。返回的是 `NodeWithScore` 对象列表。

这个功能非常实用——如果 RAG 回答质量差，先检查检索结果是否靠谱。如果检索到的内容就不对，调整 LLM 提示词也没用。

### 与 LangChain 对比

| | LangChain | LlamaIndex |
|--|-----------|------------|
| 获取检索器 | `vectorstore.as_retriever()` | `index.as_retriever()` |
| 调用方式 | `.invoke(query)` | `.retrieve(query)` |
| 返回类型 | `List[Document]` | `List[NodeWithScore]` |
| 相似度分数 | 默认不返回（需额外配置） | 默认包含 `score` 字段 |

> LlamaIndex 默认就返回相似度分数，这对调试非常友好。LangChain 需要使用 `search_type="similarity_score_threshold"` 或其他方式才能拿到分数。

## 4. 响应合成模式

检索完成后，查询引擎需要将检索结果交给 LLM 生成回答。这一步叫**响应合成（Response Synthesis）**，LlamaIndex 提供了多种模式。

核心问题：当检索到多个文档片段时，怎么把它们交给 LLM？

In [ ]:
# compact 模式（默认）：把所有检索内容塞进一个 prompt
engine_compact = index.as_query_engine(response_mode="compact", similarity_top_k=3)
response = engine_compact.query("RAG 和 Agent 有什么区别？")
print("【compact 模式】")
print(response)

In [ ]:
# refine 模式：逐个节点迭代优化答案
engine_refine = index.as_query_engine(response_mode="refine", similarity_top_k=3)
response = engine_refine.query("RAG 和 Agent 有什么区别？")
print("【refine 模式】")
print(response)

In [ ]:
# tree_summarize 模式：层次化总结
engine_tree = index.as_query_engine(response_mode="tree_summarize", similarity_top_k=3)
response = engine_tree.query("RAG 和 Agent 有什么区别？")
print("【tree_summarize 模式】")
print(response)

### 刚才发生了什么？

同一个问题，三种模式可能生成不同质量和风格的回答。它们的工作原理差异很大：

| 模式 | 工作方式 | LLM 调用次数 | 适合场景 |
|------|---------|-------------|----------|
| `compact` | 尽可能把所有内容塞进一个 prompt，超长则分批 | 1 次（通常） | 默认选择，大部分场景 |
| `refine` | 用第一个节点生成初始答案，再逐个节点迭代优化 | N 次（N=节点数） | 需要精细处理每个来源 |
| `tree_summarize` | 对节点分组总结，再对总结做总结（树状合并） | log(N) 次 | 大量文档的总结任务 |
| `simple_summarize` | 截断到上下文窗口直接生成 | 1 次 | 快速粗略回答 |

### 如何选择？

- **大部分场景用 `compact`**（默认值），省 token、速度快
- **文档很多且需要精确时用 `refine`**，但 token 消耗高
- **总结大量文档时用 `tree_summarize`**，平衡质量和效率

### 与 LangChain 对比

LangChain 的默认 RAG 链使用 "stuff" 策略（把所有文档塞进一个 prompt），相当于 LlamaIndex 的 `compact`。如果想实现 refine 或 map-reduce，需要手动用 LCEL 编排多个步骤。LlamaIndex 只需改一个参数——这是它在 RAG 场景下的优势。

## 5. 自定义提示词模板

查询引擎使用内置的提示词模板将检索内容和问题拼接后发送给 LLM。你可以替换这个模板来控制回答风格。

In [ ]:
from llama_index.core import PromptTemplate

# 自定义 QA 提示词
custom_qa_prompt = PromptTemplate(
    "你是一位 AI 技术专家。请基于以下参考内容回答问题。\n"
    "如果参考内容不足以回答，请明确说明。\n\n"
    "参考内容：\n{context_str}\n\n"
    "问题：{query_str}\n\n"
    "回答："
)

engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=custom_qa_prompt,
)
response = engine.query("什么是提示词工程？有哪些常见技巧？")
print(response)

### 刚才发生了什么？

我们用 `PromptTemplate` 创建了自定义的 QA 提示词模板，然后通过 `text_qa_template` 参数传给查询引擎。

模板中有两个必需的占位符：

| 变量 | 说明 |
|------|------|
| `{context_str}` | 检索到的文档内容，由 ResponseSynthesizer 自动填充 |
| `{query_str}` | 用户的问题 |

### 与 LangChain 对比

| | LangChain | LlamaIndex |
|--|-----------|------------|
| 模板类 | `ChatPromptTemplate` / `PromptTemplate` | `PromptTemplate` |
| 变量语法 | `{variable}` | `{variable}` |
| 使用方式 | 在 LCEL 链中显式传入 | 通过 `text_qa_template` 参数传入 |

语法几乎相同，但 LlamaIndex 的模板是专门为 RAG 场景设计的——它预设了 `context_str` 和 `query_str` 两个标准变量，而 LangChain 的模板更通用。

In [ ]:
# 查看默认的 QA 提示词模板长什么样
default_engine = index.as_query_engine()
prompts = default_engine.get_prompts()

for key, prompt in prompts.items():
    print(f"【{key}】")
    print(prompt.get_template()[:300])
    print("...\n")

### 刚才发生了什么？

`get_prompts()` 方法可以查看查询引擎当前使用的所有提示词模板。这在调试时非常有用——你能看到 LlamaIndex 默认是怎么构造 prompt 的，然后有针对性地修改。

## 6. 流式输出

对于较长的回答，流式输出可以让用户更快看到结果，提升体验。

In [ ]:
# 流式输出
engine = index.as_query_engine(streaming=True, similarity_top_k=2)
response = engine.query("解释什么是文本嵌入")

for text in response.response_gen:
    print(text, end="", flush=True)
print()

### 刚才发生了什么？

1. 设置 `streaming=True` 启用流式模式
2. `.query()` 返回的 `response` 对象包含一个 `response_gen` 生成器
3. 遍历生成器即可逐步获取文本片段

注意：流式模式下，`response.response` 在遍历 `response_gen` 之前是 `None`。必须先消费完生成器，才能通过 `response.response` 获取完整回答。

### 常见问题

**Q：流式模式下还能获取 source_nodes 吗？**

可以。`response.source_nodes` 在流式模式下依然可用，因为检索步骤在 LLM 生成之前就完成了。

## 7. 手动组装查询引擎

`as_query_engine()` 是一个便捷方法，它帮你把 Retriever 和 ResponseSynthesizer 组装在一起。如果你需要完全控制每个组件，可以手动组装。

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

# 分别创建组件
retriever = index.as_retriever(similarity_top_k=3)
synthesizer = get_response_synthesizer(
    response_mode="compact",
    text_qa_template=custom_qa_prompt,
)

# 手动组装
custom_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
)

response = custom_engine.query("什么是 RAG？")
print(response)

### 刚才发生了什么？

这段代码做的事和 `as_query_engine()` 完全一样，但把每个组件拆开了：

1. **`index.as_retriever()`** —— 创建检索器
2. **`get_response_synthesizer()`** —— 创建响应合成器，可以独立配置模式和模板
3. **`RetrieverQueryEngine()`** —— 把两者组装成查询引擎

这种方式的好处是：
- 可以给 `RetrieverQueryEngine` 传入 `node_postprocessors` 参数，添加自定义的后处理器（第五章会讲）
- 可以复用同一个 Retriever 搭配不同的 Synthesizer
- 更容易做单元测试

> **类比 LangChain**：这类似于 LangChain 中用 LCEL 手动编排 `retriever | prompt | llm | output_parser`，而非使用 `create_stuff_documents_chain` 等便捷函数。

In [ ]:
# 验证手动组装和便捷方法的结果一致
response_auto = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact",
    text_qa_template=custom_qa_prompt,
).query("什么是 RAG？")

print("【便捷方法】")
print(f"源节点数: {len(response_auto.source_nodes)}")
print(f"回答: {str(response_auto)[:100]}...")
print()
print("【手动组装】")
print(f"源节点数: {len(response.source_nodes)}")
print(f"回答: {str(response)[:100]}...")

### 刚才发生了什么？

两种方式检索到的源节点数量相同，回答风格也类似（具体内容可能因 LLM 随机性略有不同）。这证明手动组装只是 `as_query_engine()` 的展开写法，不是两种不同的机制。

选择建议：
- **快速原型 / 简单场景**：用 `as_query_engine()`，一行搞定
- **需要后处理器 / 自定义组件**：用手动组装，获得完全控制

## 总结

本章学习了：
- 查询引擎三步流程：Retriever → PostProcessor → ResponseSynthesizer
- `similarity_top_k` 控制检索数量，影响上下文丰富度和噪声
- `as_retriever()` 纯检索模式，用于调试检索效果
- 四种响应模式：compact / refine / tree_summarize / simple_summarize
- 自定义提示词模板（`text_qa_template`），使用 `{context_str}` 和 `{query_str}` 占位符
- 流式输出 `streaming=True`，通过 `response_gen` 获取增量文本
- 手动组装查询引擎（`RetrieverQueryEngine`），获得完全控制

## 下一章预告

**第四章：向量存储与持久化** —— 目前索引存在内存中，重启就丢失。下一章学习如何用 FAISS 做持久化存储，以及如何配置 Embedding 模型。